In [1]:
import numpy as np 
import faiss
import sys
import time
import csv
import os
from scipy.spatial.distance import cdist

module_path = '/home/cpanourg/projects/2-hdvc/'

if module_path not in sys.path:
    sys.path.append(module_path)

from src.utils import read_fvecs
from src.utils import append_or_create_csv

In [2]:
# Loading the GIST dataset
db = np.array(read_fvecs('/data/cpanourg/2-hdvc/data/gist/gist_base.fvecs'))
qr = np.array(read_fvecs('/data/cpanourg/2-hdvc/data/gist/gist_query.fvecs'))


Reading File - /data/cpanourg/2-hdvc/data/gist/gist_base.fvecs:(1000000, 960)
Reading File - /data/cpanourg/2-hdvc/data/gist/gist_query.fvecs:(1000, 960)


In [3]:
dataset_name = 'GIST'
sampling_method = 'random'  # 'random'
train_size_ratio = 0.1  # Ratio of the database to use for training if using sampling
nb = db.shape[0]  # Number of database vectors
nq = qr.shape[0]  # Number of query vectors
k = 100 # Number of nearest neighbors to search for
dim = db.shape[1]  # Dimensionality of the vectors

nbits = 8 # 2^nbits is the number of centroids for each subquantizer
n_subquantizers = 8   # Number of subquantizers for the PQ

In [4]:

start = time.time()

lsq = faiss.LocalSearchQuantizer(dim, n_subquantizers, nbits)


if sampling_method == 'random':
    num_samples = min(int(train_size_ratio * nb), nb)  
    training_data = db[np.random.choice(nb, num_samples, replace=False)]
    print(f"Training LSQ on {num_samples} random samples from the database.")


lsq.train(training_data)

end = time.time()

lsq_train_add_time = np.round(end - start, 2)
print(f"Time to train and add to PQ index: {lsq_train_add_time} seconds")
print("is_trained:", lsq.is_trained)

Training LSQ on 100000 random samples from the database.
Time to train and add to PQ index: 102.75 seconds
is_trained: True


In [5]:
start = time.time()
db_lsq = lsq.compute_codes(db)

end = time.time()
codes_time = np.round(end - start, 2)
print(f"Time to compute codes for the database: {codes_time} seconds")

Time to compute codes for the database: 60.6 seconds


In [6]:
start = time.time()
qr_lsq = lsq.compute_codes(qr)
end = time.time()
query_codes_time = np.round(end - start, 2)
print(f"Time to compute codes for the queries: {query_codes_time} seconds")

Time to compute codes for the queries: 0.06 seconds


In [7]:
start = time.time()
approx_dists = cdist(db_lsq, qr_lsq, metric='euclidean')
end = time.time()
approx_dist_time = np.round(end - start, 2)
print(f"Time to compute approx distances: {approx_dist_time} seconds")

Time to compute approx distances: 5.56 seconds


In [8]:
start = time.time()
exact_dists = cdist(db, qr, metric='euclidean')
end = time.time()
exact_dist_time = np.round(end - start, 2)
print(f"Time to compute exact distances: {exact_dist_time} seconds")

Time to compute exact distances: 309.92 seconds


In [12]:
np.sqrt(approx_dists)

array([[18.15246, 17.67182, 15.30522, ..., 17.62127, 14.93499, 14.86523],
       [16.78419, 17.57784, 13.94529, ..., 16.43798, 17.12362, 14.91566],
       [12.96573, 13.61585, 17.7231 , ..., 15.6742 , 16.31423, 16.66432],
       ...,
       [19.69529, 19.93175, 15.41969, ..., 14.3963 , 16.30172, 17.68028],
       [18.16986, 18.49024, 12.43653, ..., 15.28034, 16.66199, 15.54897],
       [17.78844, 19.34523, 14.02327, ..., 17.72539, 18.55875, 17.85495]])

In [ ]:
lsq.

0